# Анализ соблюдения рекомендаций тестирования детей по истории

**Задача хакатона:** создать инструмент проверки результатов тестирований — выявить нарушения частоты (не чаще 1 раза в 3 месяца), аномалии в данных и паттерны, свидетельствующие о проблемах качества данных.

**Структура ноутбука:**
1. Загрузка и первичный осмотр
2. Дедупликация (полные дубликаты)
3. Нормализация атрибутов (id_doc, variant, class, даты)
4. Дедупликация по ключу (после нормализации)
5. Точечные исправления аномалий
6. Заполнение пустых id_doc через person_key (ФИО + дата рождения)
7. Создание person_id — единого идентификатора ребёнка
8. Выявление аномалий на чистых данных
9. Сохранение результата

## 1. Загрузка и первичный осмотр

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('data/hakaton.csv', sep=';')
print(f"Исходный датасет: {len(df)} записей, {df.shape[1]} столбцов")
df.head()

In [ ]:
df.info()

In [ ]:
# Посмотрим на пропуски
df.isna().sum()

## 2. Удаление полных дубликатов

Поле `result` содержит значения в разном регистре (`Недостаточный` / `НЕДОСТАТОЧНЫЙ`), из-за чего записи-дубликаты выглядят уникальными. Приводим к верхнему регистру, после чего удаляем полные дубликаты.

Столбцы `our_number`, `name_naprav`, `name_area` не участвуют в сравнении:
- `our_number` — уникальный номер записи в системе (всегда разный)
- `name_naprav`, `name_area` — текстовые названия школ (могут различаться написанием при одном ОГРН)

In [ ]:
# Приводим результат к единому регистру
df['result'] = df['result'].str.strip().str.upper()

# Удаляем дубликаты по содержательным столбцам
cols_content = [c for c in df.columns if c not in ['our_number', 'name_naprav', 'name_area']]
before = len(df)
df = df.drop_duplicates(subset=cols_content, keep='first')
print(f"Удалено полных дубликатов: {before - len(df)}")
print(f"Осталось: {len(df)}")

# Убираем служебные столбцы — дальше они не нужны
df = df.drop(columns=['our_number', 'name_naprav', 'name_area'])

## 3. Нормализация атрибутов

### 3.1 Даты

Приводим строковые даты к `datetime`.

In [ ]:
for col in ['bdate', 'test_date', 'guard_bdate']:
    df[col] = pd.to_datetime(df[col])

print("Даты приведены к datetime")

### 3.2 Нормализация id_doc

В поле `id_doc` обнаружены проблемы:
- Одиночный символ `"-"` вместо пустого значения → заменяем на NaN
- Префиксы `№`, `N`, `-` перед числом (`№512895`, `N18005999`, `-1236233`) → убираем, оставляем только цифры

Это критически важно: без нормализации записи одного ребёнка с `id_doc = "512895"` и `id_doc = "№512895"` считаются разными людьми, и нарушения частоты между ними не ловятся.

In [ ]:
# Одиночный "-" → пропуск
df['id_doc'] = df['id_doc'].replace('-', np.nan)

def normalize_id(v):
    """Убираем из id символы №, N, -, пробелы — оставляем только цифры."""
    if pd.isna(v):
        return v
    cleaned = re.sub(r'[№N\-\s]', '', str(v).strip())
    if cleaned == '' or not any(c.isdigit() for c in cleaned):
        return np.nan
    return cleaned

df['id_doc'] = df['id_doc'].apply(normalize_id)
df['guard_id_doc'] = df['guard_id_doc'].apply(normalize_id)

print(f"id_doc после нормализации:")
print(f"  NaN: {df['id_doc'].isna().sum()}")
print(f"  Все числовые: {df['id_doc'].dropna().str.match(r'^\d+$').all()}")

### 3.3 Нормализация variant

Поле `variant` (вариант тестирования) должно быть числовым кодом, но содержит множество форматов:
- `"УЧ 120125"`, `"УЧ 120309,ПЧ 120309"` → извлекаем первое число
- `"727.110908"`, `"727/110708"` → убираем префикс `727.`
- `"№ 100807"`, `"0 80109"`, `` "`020134" `` → опечатки, убираем мусор
- `"неявился"`, даты, адреса → невосстановимые, заменяем на NaN

Нераспаршенные значения (даты вместо кодов, адрес, «неявился») — будут удалены на следующем шаге.

In [ ]:
def normalize_variant(v):
    """Извлекает числовой код варианта из различных форматов записи."""
    if pd.isna(v):
        return v
    v = str(v).strip()

    # Уже числовой — ничего не делаем
    try:
        int(v)
        return v
    except ValueError:
        pass

    # "УЧ XXXXXX" или "УЧ XXXXXX,ПЧ XXXXXX" → первое число
    m = re.match(r'^УЧ\s+(\d+)', v)
    if m: return m.group(1)

    # "727.XXXXXX", "747.XXXXXX" (разделители . _ /)
    m = re.match(r'^7[24]7[._/](\d{5,7})$', v)
    if m: return m.group(1)

    # "727.110.508" → "110508"
    m = re.match(r'^727\.(\d+)\.(\d+)$', v)
    if m: return m.group(1) + m.group(2)

    # "№ XXXXXX" или "№XXXXXX"
    m = re.match(r'^№\s*(\d+)', v)
    if m: return m.group(1)

    # "0 XXXXX" (пробел после нуля — опечатка)
    m = re.match(r'^0\s+(\d+)$', v)
    if m: return '0' + m.group(1)

    # "`XXXXXX" (бэктик — опечатка)
    m = re.match(r'^`(\d+)$', v)
    if m: return m.group(1)

    # "120.126" → "120126" (точка, но не дата)
    m = re.match(r'^(\d{3,})\.(\d+)$', v)
    if m: return m.group(1) + m.group(2)

    # Всё остальное (даты, адреса, "неявился") → NaN
    return np.nan

df['variant'] = df['variant'].apply(normalize_variant)
print(f"variant после нормализации: NaN = {df['variant'].isna().sum()}")

### 3.4 Нормализация class

Одна запись содержит значение `"2-5"` — невалидный класс. По возрасту ребёнка (11 лет) и варианту тестирования заменяем на `"3"`.

In [ ]:
df.loc[df['class'] == '2-5', 'class'] = '3'
print("class='2-5' → '3'")

## 4. Дедупликация по ключу (после нормализации id_doc)

После нормализации id_doc записи вроде `"512895"` и `"№512895"` стали одинаковыми — можно найти дополнительные дубликаты.

Ключ дедупликации: `(id_doc, test_date, variant, class)` — один ребёнок не может сдавать один и тот же вариант за один класс в один день дважды.

Записи без id_doc (NaN) не участвуют в дедупликации — их обработаем позже.

In [ ]:
mask_valid = df['id_doc'].notna()
df_v = df[mask_valid].copy()
df_inv = df[~mask_valid].copy()

before = len(df_v)
df_v = df_v.sort_values(['id_doc', 'test_date', 'variant', 'class', 'result'])
df_v = df_v.drop_duplicates(subset=['id_doc', 'test_date', 'variant', 'class'], keep='last')
print(f"Дубликаты по ключу (id_doc, date, variant, class): {before - len(df_v)}")

df = pd.concat([df_v, df_inv], ignore_index=True)
print(f"Итого: {len(df)}")

## 5. Точечные исправления аномалий

Здесь обрабатываем конкретные случаи, выявленные при ручном анализе данных.

### 5.1 Удаление записи с некорректным ОГРН

Одна запись имеет `ogrn_naprav` длиной 15 символов вместо стандартных 13 — невалидный ОГРН.

In [ ]:
mask_ogrn = df['ogrn_naprav'].astype(str).str.len() != 13
print(f"Записей с некорректным ОГРН: {mask_ogrn.sum()}")
df = df[~mask_ogrn]

### 5.2 Удаление записей с нераспаршенным вариантом

Записи, где variant не удалось привести к числовому коду (даты вместо кодов, адрес в поле варианта, `"неявился"` и т.д.).

In [ ]:
mask_var = df['variant'].isna()
print(f"Записей с невалидным вариантом: {mask_var.sum()}")
df = df[~mask_var]

### 5.3 Исправление дат рождения

У нескольких детей дата рождения явно ошибочна (год 2025 вместо реального). У каждого из них есть другие записи с тем же id_doc и правильной датой — по ним восстанавливаем.

| Ребёнок | id_doc | Было | Стало | Обоснование |
|---------|--------|------|-------|-------------|
| ЛУКОНИН МИХАИЛ | 13653549 | 2025-09-18 | 2012-09-18 | Другие записи того же id — 2012 |
| ЗАХАРОВ АЛЕКСЕЙ | 0384771 | 2025-09-14 | 2018-09-14 | Другие записи — 2018 |
| СОРОКИНА МАРИЯ | 18151280 | 2025-05-16 | 2015-05-16 | Другие записи — 2015 |
| КОМАРДИНА ИРИНА | 405348032 | 2025-08-13 | 2011-08-13 | Другие записи — 2011, предположение: близнецы |

In [ ]:
# Исправляем bdate по данным из других записей того же ребёнка
fixes_bdate = [
    ('13653549', '2025-09-18', '2012-09-18'),   # ЛУКОНИН МИХАИЛ
    ('0384771',  '2025-09-14', '2018-09-14'),   # ЗАХАРОВ АЛЕКСЕЙ
    ('18151280', '2025-05-16', '2015-05-16'),   # СОРОКИНА МАРИЯ
    ('405348032','2025-08-13', '2011-08-13'),   # КОМАРДИНА ИРИНА
]

for id_doc, old_date, new_date in fixes_bdate:
    mask = (df['id_doc'] == id_doc) & (df['bdate'] == pd.Timestamp(old_date))
    df.loc[mask, 'bdate'] = pd.Timestamp(new_date)

print("Даты рождения исправлены")

### 5.4 Исправление guard_same_age

У трёх детей `bdate == guard_bdate` — дата рождения ребёнка скопирована с даты рождения родителя. Других записей для этих id нет, поэтому восстанавливаем год рождения по формуле: `год теста - класс - 6`.

| Ребёнок | id_doc | bdate (было) | class | bdate (стало) |
|---------|--------|-------------|-------|---------------|
| КУЗИНА ВИКТОРИЯ | 0535794 | 2006-01-23 | 9 | 2011-01-23 |
| ТОКНЕШЕВ ОЛЕГ | 1306476 | 1996-10-04 | 10 | 2009-10-04 |
| НЕФЕДОВ ДМИТРИЙ | 576809 | 1992-02-27 | 1 | 2018-02-27 |

In [ ]:
# guard_same_age: bdate скопирован с родителя → восстанавливаем по классу
guard_fixes = [
    ('0535794', '2006-01-23', '2011-01-23'),   # КУЗИНА ВИКТОРИЯ, class=9
    ('1306476', '1996-10-04', '2009-10-04'),   # ТОКНЕШЕВ ОЛЕГ, class=10
    ('576809',  '1992-02-27', '2018-02-27'),   # НЕФЕДОВ ДМИТРИЙ, class=1
]

for id_doc, old_date, new_date in guard_fixes:
    mask = (df['id_doc'] == id_doc) & (df['bdate'] == pd.Timestamp(old_date))
    df.loc[mask, 'bdate'] = pd.Timestamp(new_date)

print("guard_same_age: даты рождения восстановлены по классу")

### 5.5 Удаление child_too_old

После исправления guard_same_age остаются записи, где ребёнку >20 лет на момент теста (БАРИНОВ 20 лет в 10 классе, КОРЧАГИН 22 года в 6 классе, КУШНИР 27 лет в 3 классе). Данных для восстановления нет — удаляем.

In [ ]:
df['child_age'] = (df['test_date'] - df['bdate']).dt.days / 365.25
mask_old = df['child_age'] > 20
print(f"child_too_old (>20 лет): удалено {mask_old.sum()}")
df = df[~mask_old]

### 5.6 Удаление child_too_young без данных

МИШИН РОМАН — bdate=2023, class=5 (возраст 2.5 года). Нет других записей с этим id_doc для восстановления.

БАБАЕВА ОЛЬГА (4.7 года, 1 класс) — оставляем с предположением, что дошкольник.  
КОМАРДИНА ИРИНА — уже исправлена выше.

In [ ]:
mask_mishin = (df['id_doc'] == '15238017') & (df['bdate'].dt.year == 2023)
print(f"МИШИН РОМАН: удалено {mask_mishin.sum()}")
df = df[~mask_mishin]

df = df.drop(columns=['child_age']).reset_index(drop=True)
print(f"\nПосле всех точечных фиксов: {len(df)} записей")
print(f"  id_doc NaN: {df['id_doc'].isna().sum()}")

## 6. Заполнение пустых id_doc

Вместо удаления записей без id_doc — заполняем их:
1. Создаём `person_key` = ФИО + дата рождения
2. Ищем совпадения с записями, у которых id_doc есть → присваиваем существующий id
3. Оставшимся — генерируем новый уникальный id

In [ ]:
# Создаём ключ физического лица: ФИО + дата рождения
df['person_key'] = (
    df['last_name'].str.strip() + '|' +
    df['first_name'].str.strip() + '|' +
    df['middle_name'].fillna('').str.strip() + '|' +
    df['bdate'].dt.strftime('%Y-%m-%d')
)

# Сколько пустых id_doc и есть ли для них совпадения?
na_mask = df['id_doc'].isna()
has_id = df[df['id_doc'].notna()]
print(f"Записей без id_doc: {na_mask.sum()}")

In [ ]:
# 1) Заполняем существующим id, если находим совпадение по person_key
id_map = has_id.groupby('person_key')['id_doc'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
)

filled = 0
for idx in df[na_mask].index:
    key = df.at[idx, 'person_key']
    if key in id_map.index:
        df.at[idx, 'id_doc'] = id_map[key]
        filled += 1

print(f"Заполнено существующим id (совпадение по ФИО+ДР): {filled}")

# 2) Остальным — генерируем новый id
still_na = df['id_doc'].isna()
remaining_keys = df.loc[still_na, 'person_key'].unique()
max_existing = pd.to_numeric(df['id_doc'].dropna(), errors='coerce').max()
new_id_start = int(max_existing) + 1

key_to_new = {}
counter = new_id_start
for key in remaining_keys:
    key_to_new[key] = str(counter)
    counter += 1

for idx in df[still_na].index:
    df.at[idx, 'id_doc'] = key_to_new[df.at[idx, 'person_key']]

print(f"Сгенерировано новых id: {len(key_to_new)}")
print(f"id_doc NaN после заполнения: {df['id_doc'].isna().sum()}")

## 7. Person ID — единый идентификатор ребёнка

Один и тот же ребёнок может фигурировать с разными id_doc (ошибки ввода, разные документы). `person_id` — это «канонический» id_doc для каждого ФИО+ДР (самый частый из имеющихся).

Это нужно для корректного подсчёта нарушений частоты: без объединения тесты одного ребёнка с разными id_doc не считаются повторными.

In [ ]:
# Для каждого person_key берём самый частый id_doc
id_map_full = df.groupby('person_key')['id_doc'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
)
df['person_id'] = df['person_key'].map(id_map_full)

# Статистика объединения
multi_id = df.groupby('person_key')['id_doc'].nunique()
merged = multi_id[multi_id > 1]

print(f"Уникальных person_key (ФИО+ДР): {df['person_key'].nunique()}")
print(f"Уникальных id_doc: {df['id_doc'].nunique()}")
print(f"Уникальных person_id: {df['person_id'].nunique()}")
print(f"Людей с >1 id_doc (объединены через person_id): {len(merged)}")

## 8. Выявление аномалий на чистых данных

### 8.1 Нарушения частоты тестирования

Рекомендация: не чаще 1 раза в 3 месяца (90 дней). Считаем по `person_id`, чтобы учесть случаи с разными id_doc у одного ребёнка.

In [ ]:
df_s = df.sort_values(['person_id', 'test_date']).copy()
df_s['prev_test'] = df_s.groupby('person_id')['test_date'].shift(1)
df_s['days_gap'] = (df_s['test_date'] - df_s['prev_test']).dt.days

freq_mask = df_s['days_gap'].notna() & (df_s['days_gap'] < 90)
df['freq_violation'] = df.index.isin(df_s[freq_mask].index)
df['same_day_test'] = df.index.isin(df_s[df_s['days_gap'] == 0].index)

print(f"Нарушения частоты (<90 дней): {df['freq_violation'].sum()} записей, {df_s[freq_mask]['person_id'].nunique()} детей")
print(f"Из них в один день: {df['same_day_test'].sum()}")

### 8.2 Скачки класса (multi_class)

Тестирования проводятся чуть больше года — ребёнок не может перескочить через класс. Нормально: 1→2 или 5→5. Аномалия: 1→7, 9→4, 6→8 в один день.

In [ ]:
df['class_num'] = pd.to_numeric(df['class'], errors='coerce')

def check_multi_class(group):
    """Есть ли у ребёнка скачки класса > 1 между последовательными тестами?"""
    g = group.sort_values('test_date')
    classes = g['class_num'].dropna().values
    if len(classes) < 2:
        return False
    for i in range(1, len(classes)):
        if abs(classes[i] - classes[i-1]) > 1:
            return True
    return False

mc_persons = df.groupby('person_id').apply(check_multi_class)
mc_ids = mc_persons[mc_persons].index
df['multi_class_jump'] = df['person_id'].isin(mc_ids)

print(f"Скачки класса (>1): {df['multi_class_jump'].sum()} записей, {len(mc_ids)} детей")

### 8.3 Прочие аномалии

- **id_doc == guard_id_doc** — ID ребёнка совпадает с ID родителя (оператор скопировал)
- **Несоответствие возраст-класс** — разница между реальным возрастом и ожидаемым (класс + 6) больше 3 лет
- **Слишком молодой родитель** — разница возраста родитель-ребёнок < 14 лет

In [ ]:
# id_doc == guard_id_doc
df['id_equals_guard'] = df['id_doc'] == df['guard_id_doc']
print(f"id_doc == guard_id_doc: {df['id_equals_guard'].sum()}")

# Несоответствие возраст-класс (>3 лет)
df['child_age'] = (df['test_date'] - df['bdate']).dt.days / 365.25
expected = df['class_num'] + 6
df['age_class_mismatch'] = ((df['child_age'] - expected).abs() > 3) & expected.notna()
print(f"Несоответствие возраст-класс (>3 лет): {df['age_class_mismatch'].sum()}")

# Родитель слишком молод
df['parent_age_at_birth'] = (df['bdate'] - df['guard_bdate']).dt.days / 365.25
df['guard_too_young'] = (df['parent_age_at_birth'] > 0) & (df['parent_age_at_birth'] < 14)
print(f"Родитель слишком молод (<14 лет): {df['guard_too_young'].sum()}")

## 9. Итоговая сводка и сохранение

In [ ]:
anomaly_cols = ['freq_violation', 'same_day_test', 'multi_class_jump',
                'id_equals_guard', 'age_class_mismatch', 'guard_too_young']
df['has_anomaly'] = df[anomaly_cols].any(axis=1)

print(f"Исходных записей: 25628")
print(f"Итоговых записей: {len(df)}")
print(f"Уникальных детей (person_id): {df['person_id'].nunique()}")
print(f"")
print(f"Записей с аномалиями: {df['has_anomaly'].sum()} ({df['has_anomaly'].mean()*100:.1f}%)")
print(f"Чистых записей: {(~df['has_anomaly']).sum()}")
print(f"")
print("По типам аномалий:")
for col in anomaly_cols:
    print(f"  {col}: {df[col].sum()}")

In [ ]:
# Сохраняем — убираем вспомогательные столбцы
export_cols = [c for c in df.columns if c not in [
    'person_key', 'class_num', 'child_age', 'parent_age_at_birth'
]]
df[export_cols].to_csv('clean_final.csv', index=False)
print(f"Сохранено: clean_final.csv ({len(df)} записей)")